# 🎙️ Speech Transcription & Audio Analysis

**Author:** Marcio da Costa Oliveira  
**GitHub:** [MARCIO-COSTA93](https://github.com/MARCIO-COSTA93)  
**LinkedIn:** [marcio-costa-oliveira](https://linkedin.com/in/marcio-costa-oliveira)

---

## Overview

This notebook demonstrates a complete pipeline for **speech-to-text transcription** using the  library, with additional audio preprocessing, data augmentation and analysis techniques relevant to real-world voice applications.

### Key features:
- Multi-format audio support: WAV, OGG, OPUS, MP3, FLAC
- Ambient noise adjustment for improved transcription accuracy
- Data augmentation: noise injection and time stretching
- Visual analysis: waveform and spectrogram before/after preprocessing
- Comparison between online (Google Web Speech API) and offline (CMU Sphinx) recognition
- Real-world use case: WhatsApp voice message transcription

### Why this matters for KWS/ASR systems:
Understanding how raw audio is preprocessed and transcribed is foundational for building **Keyword Spotting (KWS)** and **Automatic Speech Recognition (ASR)** pipelines. Techniques like noise adjustment and format normalization are essential steps before any deep learning model can reliably detect keywords or transcribe speech.

## 1. Installing Dependencies

In [ ]:
# SpeechRecognition: main library for speech-to-text
# PyAudio: required for microphone input
# pydub: audio format conversion (OGG, MP3, etc.)
# audiomentations: state-of-the-art audio data augmentation
!pip install SpeechRecognition PyAudio pydub audiomentations
!apt-get install -y ffmpeg  # required for format conversion

## 2. Importing Libraries

In [ ]:
import os
import numpy as np
import librosa
import librosa.display as ld
import matplotlib.pyplot as plt
import soundfile as sf
import speech_recognition as sr
from pydub import AudioSegment
from IPython.display import Audio, display
from audiomentations import Compose, AddGaussianNoise, TimeStretch, PitchShift

print(f"SpeechRecognition version: {sr.__version__}")
print(f"librosa version: {librosa.__version__}")

## 3. Audio Utility Functions

Centralizing reusable functions is a best practice for maintainable ML pipelines.

In [ ]:
def load_audio(file_path, target_sr=16000):
    """
    Load an audio file and resample to target sample rate.

    Why 16kHz? Most speech recognition models are trained on 16kHz audio.
    Resampling ensures compatibility across different input sources.

    Args:
        file_path (str): Path to audio file
        target_sr (int): Target sample rate in Hz (default: 16000)

    Returns:
        tuple: (audio_data, sample_rate)
    """
    data, sr_rate = librosa.load(file_path, sr=target_sr)
    print(f"File: {os.path.basename(file_path)}")
    print(f"Sample rate: {sr_rate} Hz")
    print(f"Duration: {len(data)/sr_rate:.2f}s")
    print(f"Total samples: {len(data)}")
    return data, sr_rate


def plot_audio_analysis(data, sr_rate, title="Audio Analysis"):
    """
    Plot waveform and spectrogram side by side.

    Waveform shows amplitude over time (energy of the signal).
    Spectrogram shows frequency content over time — this is what
    neural networks see when processing audio.
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    # Waveform
    axes[0].set_title("Waveform")
    ld.waveshow(data, sr=sr_rate, ax=axes[0])
    axes[0].set_xlabel("Time (s)")
    axes[0].set_ylabel("Amplitude")

    # STFT Spectrogram
    # STFT decomposes the signal into frequency components over time
    # This is the visual representation CNNs/RNNs process in KWS systems
    stft = librosa.stft(data)
    stft_db = librosa.amplitude_to_db(np.abs(stft))
    img = ld.specshow(stft_db, sr=sr_rate, x_axis="time", y_axis="log", ax=axes[1])
    axes[1].set_title("Spectrogram (STFT)")
    fig.colorbar(img, ax=axes[1], format="%+2.0f dB")

    plt.tight_layout()
    plt.show()


def convert_to_wav(input_path, output_path=None):
    """
    Convert audio file to WAV format.

    SpeechRecognition natively supports WAV. Converting other formats
    (OGG, OPUS, MP3) ensures compatibility with all recognition engines.
    """
    ext = os.path.splitext(input_path)[1].lower().replace(".", "")
    if output_path is None:
        output_path = os.path.splitext(input_path)[0] + "_converted.wav"

    format_map = {"ogg": "ogg", "opus": "opus", "mp3": "mp3", "flac": "flac", "wav": "wav"}
    audio_format = format_map.get(ext, "wav")

    audio = AudioSegment.from_file(input_path, format=audio_format)
    audio.export(output_path, format="wav")
    print(f"Converted {ext.upper()} to WAV: {output_path}")
    return output_path

## 4. Data Augmentation

Data augmentation artificially increases dataset diversity by applying transformations to existing audio. This is critical for building robust speech models that work in real-world conditions (background noise, different speaking speeds, etc.).

In [ ]:
def apply_augmentation(data, sr_rate, augmentation_type="all"):
    """
    Apply audio data augmentation techniques.

    Augmentation types:
    - noise: Adds Gaussian noise — simulates real-world background noise
    - stretch: Time stretching — simulates different speaking speeds
    - pitch: Pitch shifting — simulates different voices/microphones
    - all: Applies all transformations
    """
    augmentations = {
        "noise": Compose([AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.015, p=1.0)]),
        "stretch": Compose([TimeStretch(min_rate=0.8, max_rate=1.2, p=1.0)]),
        "pitch": Compose([PitchShift(min_semitones=-4, max_semitones=4, p=1.0)]),
        "all": Compose([
            AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.015, p=0.5),
            TimeStretch(min_rate=0.8, max_rate=1.2, p=0.5),
            PitchShift(min_semitones=-4, max_semitones=4, p=0.5)
        ])
    }
    augment = augmentations.get(augmentation_type, augmentations["all"])
    return augment(samples=data, sample_rate=sr_rate)


def compare_augmentation(data, sr_rate, augmentation_type="noise"):
    """
    Visualize original vs augmented audio side by side.
    """
    augmented = apply_augmentation(data, sr_rate, augmentation_type)

    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    fig.suptitle(f"Augmentation: {augmentation_type}", fontsize=14, fontweight="bold")

    ld.waveshow(data, sr=sr_rate, ax=axes[0][0])
    axes[0][0].set_title("Original - Waveform")

    ld.waveshow(augmented, sr=sr_rate, ax=axes[0][1])
    axes[0][1].set_title(f"Augmented ({augmentation_type}) - Waveform")

    stft_orig = librosa.amplitude_to_db(np.abs(librosa.stft(data)))
    ld.specshow(stft_orig, sr=sr_rate, x_axis="time", y_axis="log", ax=axes[1][0])
    axes[1][0].set_title("Original - Spectrogram")

    stft_aug = librosa.amplitude_to_db(np.abs(librosa.stft(augmented)))
    ld.specshow(stft_aug, sr=sr_rate, x_axis="time", y_axis="log", ax=axes[1][1])
    axes[1][1].set_title(f"Augmented ({augmentation_type}) - Spectrogram")

    plt.tight_layout()
    plt.show()
    return augmented

## 5. Speech Recognition

Comparing online (Google) vs offline (Sphinx) recognition engines.
In production KWS systems, offline recognition is preferred for privacy and latency.

In [ ]:
recognizer = sr.Recognizer()


def transcribe_audio(file_path, engine="google", language="pt-BR", adjust_noise=True, noise_duration=0.5):
    """
    Transcribe audio file using specified recognition engine.

    Args:
        file_path (str): Path to WAV audio file
        engine (str): "google" (online) or "sphinx" (offline)
        language (str): Language code (e.g. "pt-BR", "en-US")
        adjust_noise (bool): Whether to adjust for ambient noise
        noise_duration (float): Seconds used to sample ambient noise

    Note on noise adjustment:
        adjust_for_ambient_noise() samples the first N seconds to estimate
        background noise, then sets an energy threshold above which audio
        is considered speech. Critical for reducing false positives in KWS.
    """
    audio_file = sr.AudioFile(file_path)
    with audio_file as source:
        if adjust_noise:
            recognizer.adjust_for_ambient_noise(source, duration=noise_duration)
        audio_data = recognizer.record(source)

    try:
        if engine == "google":
            result = recognizer.recognize_google(audio_data, language=language)
        elif engine == "sphinx":
            result = recognizer.recognize_sphinx(audio_data)
        else:
            result = "Unknown engine."
        print(f"[{engine.upper()}] {result}")
        return result
    except sr.UnknownValueError:
        print(f"[{engine.upper()}] Could not understand audio.")
        return None
    except sr.RequestError as e:
        print(f"[{engine.upper()}] API error: {e}")
        return None


def compare_engines(file_path, language="pt-BR"):
    """Compare Google (online) vs Sphinx (offline) transcription."""
    print("=" * 50)
    print("ENGINE COMPARISON")
    print("=" * 50)
    google_result = transcribe_audio(file_path, engine="google", language=language)
    sphinx_result = transcribe_audio(file_path, engine="sphinx")
    print(f"
Google : {google_result}")
    print(f"Sphinx : {sphinx_result}")
    return google_result, sphinx_result

## 6. Full Pipeline

Combining all steps: load → convert → visualize → augment → transcribe.

In [ ]:
def process_audio_file(file_path, language="pt-BR", show_analysis=True):
    """
    Full pipeline: load -> convert -> visualize -> augment -> transcribe.

    This demonstrates the complete preprocessing pipeline that would
    precede a keyword detection model in production.
    """
    ext = os.path.splitext(file_path)[1].lower()
    print(f"Processing: {os.path.basename(file_path)} [{ext}]")

    # Step 1: Convert to WAV if needed
    wav_path = convert_to_wav(file_path) if ext != ".wav" else file_path

    # Step 2: Load and analyze
    data, sample_rate = load_audio(wav_path)
    display(Audio(data=data, rate=sample_rate))

    if show_analysis:
        plot_audio_analysis(data, sample_rate, title=f"Original: {os.path.basename(file_path)}")

    # Step 3: Augmentation comparison
    if show_analysis:
        print("
Applying noise augmentation...")
        compare_augmentation(data, sample_rate, augmentation_type="noise")

    # Step 4: Transcribe
    print("
Transcribing...")
    return transcribe_audio(wav_path, language=language)


# Example — replace with your own audio file:
# process_audio_file("/content/audios/your_audio.wav", language="pt-BR")

## 7. WhatsApp Voice Message Batch Transcription

In [ ]:
def transcribe_whatsapp_messages(audio_dir, language="pt-BR"):
    """
    Batch transcription of WhatsApp voice messages (OGG/OPUS).
    Converts all files in a directory and transcribes them.
    """
    supported = [".ogg", ".opus", ".mp3", ".wav", ".flac"]
    files = [f for f in os.listdir(audio_dir) if os.path.splitext(f)[1].lower() in supported]
    print(f"Found {len(files)} file(s)")
    results = {}
    for filename in files:
        path = os.path.join(audio_dir, filename)
        ext = os.path.splitext(filename)[1].lower()
        wav_path = convert_to_wav(path) if ext != ".wav" else path
        results[filename] = transcribe_audio(wav_path, language=language)
    print("
Results:")
    for k, v in results.items():
        print(f"  {k}: {v}")
    return results

# Example:
# transcribe_whatsapp_messages("/content/audios/", language="pt-BR")

## 8. Observations & Conclusions

### Key Takeaways

**Preprocessing matters as much as the model:**  
Ambient noise adjustment alone significantly impacts transcription quality. The same principle applies to Keyword Spotting systems — a well-calibrated energy threshold reduces false positives dramatically.

**Online vs Offline trade-offs:**  
- Google Web Speech API: high accuracy, requires internet, latency ~500ms  
- CMU Sphinx: lower accuracy, no internet required, runs on embedded devices  
- For KWS on edge devices, offline inference is preferred for latency and privacy

**Data Augmentation for robustness:**  
Gaussian noise injection and time stretching simulate real-world variability. Models trained with augmented data generalize better — critical for always-on keyword detection systems.

### Connection to Keyword Spotting
The preprocessing pipeline built here — resampling to 16kHz, noise calibration, spectrogram generation — is the exact same foundation used in KWS systems like wake word detectors.